In [9]:
def validate_design_matrix(X):
    import sympy
    M = sympy.Matrix(X.astype(int).tolist())
    null_vecs = M.nullspace()
    rank = X.shape[1] - len(null_vecs)
    if rank == X.shape[1]:
        return True
    col_names = X.design_info.column_names
    print(f"Design matrix rank: {rank}, shape: {X.shape}")
    print("Linear dependencies:")
    for vec in null_vecs:
        terms = []
        for j, coef in enumerate(vec):
            if coef != 0:
                col = col_names[j]
                if coef.is_Integer:
                    coef = int(coef)
                    if coef == 1:
                        terms.append(col)
                    elif coef == -1:
                        terms.append(f"-{col}")
                    else:
                        terms.append(f"{coef}*{col}")
                else:
                    terms.append(f"{coef}*{col}")
        print(f"  {' + '.join(terms).replace('+ -', '- ')} = 0")
    return False

In [54]:
import pandas as pd
import patsy
import statsmodels.api as sm
import numpy as np

def add_safe_interactions(df, interactions):
    new_cols = []
    for var1, var2 in interactions:
        for level in df[var1].unique():
            mask = df[var1] == level
            if df.loc[mask, var2].nunique() > 1:
                col_name = f"{var1}_{level}_{var2}"
                df[col_name] = mask & df[var2]
                new_cols.append(col_name)
    return new_cols

data = pd.DataFrame({
    'student': ['A', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'C', 'C', 'C', 'C'],
    'subject': ['Math', 'Math', 'Physics', 'Physics', 'Math', 'Math', 'Physics', 'Physics', 'Physics', 'Physics', 'Math', 'Physics'],
    'studied': [True, False, True, False, True, True, True, True, True, False, True, False],
    'test_score': [90, 70, 85, 80, 75, 65, 95, 70, 20, 30, 40, 50]
})

interaction_cols = add_safe_interactions(data, [('student', 'studied')])
y, X = patsy.dmatrices(f"test_score ~ C(student, Sum) + C(subject, Sum) + {' + '.join(interaction_cols)}", data)

print(f"Is design Matrix valid? {validate_design_matrix(X)}")

print(pd.DataFrame(X, columns=X.design_info.column_names).to_string())
print()
print()
print(interaction_cols)
print()

model = sm.OLS(y, X)
result = model.fit()
print(result.summary())
print("\n")

# Create final dataframe with all terms, coefficients, and standard errors
params = result.params
cov = result.cov_params()
param_names = X.design_info.column_names

# Base dataframe with explicit terms
final_df = pd.DataFrame({
    'term': param_names,
    'coeff': params,
    'std_err': np.sqrt(np.diag(cov))
})

# Add implied terms from sum contrasts
# Student C (implied from sum contrast on student)
a_idx = param_names.index('C(student, Sum)[S.A]')
b_idx = param_names.index('C(student, Sum)[S.B]')
coeff_c = -(params[a_idx] + params[b_idx])
se_c = np.sqrt(cov[a_idx, a_idx] + cov[b_idx, b_idx] + 2 * cov[a_idx, b_idx])
final_df.loc[len(final_df)] = ['Student C', coeff_c, se_c]

# Subject Physics (implied from sum contrast on subject)
math_idx = param_names.index('C(subject, Sum)[S.Math]')
coeff_physics = -params[math_idx]
se_physics = np.sqrt(cov[math_idx, math_idx])
final_df.loc[len(final_df)] = ['Subject Physics', coeff_physics, se_physics]

# Clean up term names for readability
final_df['term'] = final_df['term'].str.replace(r'\[T\.True\]', '', regex=True)
final_df['term'] = final_df['term'].str.replace(r'C\(student, Sum\)\[S\.', 'Student ', regex=True)
final_df['term'] = final_df['term'].str.replace(r'C\(subject, Sum\)\[S\.', 'Subject ', regex=True)
final_df['term'] = final_df['term'].str.replace(r'\]', '', regex=True)
final_df['term'] = final_df['term'].str.replace(r'student_(\w+)_studied', r'Student \1 × Studied', regex=True)
final_df['term'] = final_df['term'].str.replace('Intercept', 'Intercept (Grand Mean)')

# Sort alphabetically
final_df = final_df.sort_values('term').reset_index(drop=True)

print(final_df)

Is design Matrix valid? True
    Intercept  C(student, Sum)[S.A]  C(student, Sum)[S.B]  C(subject, Sum)[S.Math]  student_A_studied[T.True]  student_C_studied[T.True]
0         1.0                   1.0                   0.0                      1.0                        1.0                        0.0
1         1.0                   1.0                   0.0                      1.0                        0.0                        0.0
2         1.0                   1.0                   0.0                     -1.0                        1.0                        0.0
3         1.0                   1.0                   0.0                     -1.0                        0.0                        0.0
4         1.0                   0.0                   1.0                      1.0                        0.0                        0.0
5         1.0                   0.0                   1.0                      1.0                        0.0                        0.0
6         1.